# Khảo sát Constraint Programming — OPL, DOcplex.cp và OR-Tools

Báo cáo so sánh cách triển khai Constraint Programming trên ba cách tiếp cận, qua
sáu bài toán thuộc các dạng cấu trúc khác nhau.

| Chiều | Ngôn ngữ mô hình hoá | Engine giải |
|---|---|---|
| **OPL** | OPL (`.mod`) | CP Optimizer — IBM |
| **DOcplex.cp** | Python, `docplex.cp` | CP Optimizer — IBM |
| **OR-Tools** | Python, `ortools.sat` | CP-SAT — Google |

---

## Vì sao phải có ba chiều chứ không phải hai

Đây là luận điểm phương pháp của toàn bộ báo cáo.

Nếu chỉ so **OPL với OR-Tools**, hai đối tượng khác nhau ở *hai* thứ cùng lúc:
ngôn ngữ mô hình hoá **và** engine giải. Khi thấy một bên nhanh hơn, ta không có
cách nào biết công lao thuộc về ngôn ngữ hay thuộc về engine. Mọi kết luận rút ra
đều lẫn hai nguyên nhân.

Chiều **DOcplex.cp** là cầu nối tách bạch hai nguyên nhân đó:

```
   OPL                DOcplex.cp                OR-Tools
   ngôn ngữ: OPL      ngôn ngữ: Python          ngôn ngữ: Python
   engine: CP Opt.    engine: CP Optimizer      engine: CP-SAT
        │                   │      │                  │
        └───────┬───────────┘      └────────┬─────────┘
                │                           │
        engine GIỐNG nhau            ngôn ngữ GIỐNG nhau
        ⇒ chênh lệch là do          ⇒ chênh lệch là do
             NGÔN NGỮ                     ENGINE
```

Mỗi phép so chỉ đổi **đúng một** biến số. Đây là điều mà so sánh trực tiếp
"OPL vs OR-Tools" không làm được.

> **Bài 3.2 cho thấy điều này không phải lý thuyết suông.** Ở đó, so OPL với
> OR-Tools cho chênh lệch 8.60 s so với 2.42 s — tưởng như "CP-SAT nhanh hơn".
> Nhưng thêm điểm đo DOcplex.cp (1.50 s) thì lộ ra sự thật khác hẳn: phần lớn
> chênh lệch đến từ **cách mã hoá**, không phải từ engine; và trên cùng cách mã
> hoá thì CP Optimizer mới là bên nhanh hơn.


## Kiến trúc tầng — phân biệt engine và ngôn ngữ mô hình hoá

Nhầm lẫn phổ biến nhất khi đọc tài liệu IBM là coi **CPLEX** và **DOcplex** như
hai lựa chọn ngang hàng. Chúng ở hai tầng khác nhau.

```
  TẦNG NGÔN NGỮ MÔ HÌNH HOÁ
  ┌───────────┐          ┌──────────────────────────┐     ┌──────────────┐
  │    OPL    │          │         DOcplex          │     │   OR-Tools   │
  │  (.mod)   │          │        (Python)          │     │   (Python)   │
  └─────┬─────┘          └───┬──────────────────┬───┘     └──────┬───────┘
        │                    │                  │                │
        │            docplex.mp          docplex.cp               │
        │                    │                  │                │
  ══════╪════════════════════╪══════════════════╪════════════════╪══════
        │                    │                  │                │
  TẦNG ENGINE GIẢI           ▼                  ▼                ▼
        │            ┌──────────────┐   ┌──────────────┐  ┌──────────────┐
        ├───────────►│    CPLEX     │   │ CP Optimizer │  │    CP-SAT    │
        │            │  LP/MILP/QP  │   │      CP      │  │      CP      │
        └───────────────────────────────►──────────────┘  └──────────────┘
                     └──── của IBM, trong CPLEX Optimization Studio ────┘   Google
```

- **CPLEX** là engine Math Programming. **CP Optimizer** là engine Constraint
  Programming. Cả hai đều của IBM, đi kèm trong CPLEX Optimization Studio.
- **OPL gọi được cả hai engine** — chính dòng `using CP;` ở đầu file `.mod` quyết
  định model chạy trên engine nào. Thiếu dòng đó, model rơi vào context CPLEX và
  báo lỗi kiểu *"Function allDifferent(...) not available in context CPLEX"*.
- **DOcplex cũng gọi được cả hai**, qua `docplex.mp` (→ CPLEX) và `docplex.cp`
  (→ CP Optimizer).
- **DOcplex không thay thế CPLEX.** Nó là thư viện Python để *viết* mô hình rồi
  gọi xuống engine. `pip install docplex` không kèm solver nào cả.

> **Phạm vi báo cáo:** chỉ nhánh CP — **OPL (CP) + DOcplex.cp + OR-Tools (CP-SAT)**.
> `docplex.mp` là paradigm khác (quy hoạch tuyến tính nguyên) và **không** xuất
> hiện ở bất kỳ đâu trong báo cáo này.


## Bản đồ phân loại bài toán CP

CP phân loại theo **cấu trúc bài toán**, không theo tuyến tính hay phi tuyến như
quy hoạch toán học. Năm dạng lớn:

| # | Dạng | Đặc trưng | Ví dụ |
|---|---|---|---|
| 1 | **Assignment / Labeling** | gán giá trị rời rạc, ràng buộc phân biệt | tô màu đồ thị, N-Queens, Sudoku |
| 2 | **Scheduling** | có chiều thời gian; chia ba nhánh con: machine scheduling, project scheduling, personnel rostering | job-shop, RCPSP, xếp ca |
| 3 | **Packing / Partitioning** | nhóm phần tử vào thùng | bin packing, steel mill |
| 4 | **Routing / Sequencing** | thứ tự và lộ trình | TSP, VRP, car sequencing |
| 5 | **Satisfaction thuần** | chỉ tìm cấu hình hợp lệ | magic square, các loại puzzle |

**Thế mạnh khác nhau của hai engine:**

- **CP Optimizer** mạnh nhất ở **scheduling với biến interval** — hơn 20 năm phát
  triển chuyên biệt. Nó có sẵn `intervalVar`, `noOverlap`, `alternative`,
  `forbidExtent` + hàm bậc thang, `cumulFunction`. Bài 2.1 và 3.2 khai thác đúng
  chỗ này.
- **CP-SAT** mạnh nhất ở **bài tổ hợp/boolean nặng** nhờ kiến trúc lai SAT với
  lazy clause generation (vô địch MiniZinc Challenge nhiều năm liền), và có thêm
  `AddAutomaton`, `AddCircuit` cùng module routing riêng mà CP Optimizer không có sẵn.

Sáu bài toán của báo cáo trải trên bản đồ này:

| Bài | Tên | Dạng |
|---|---|---|
| 1.1 | Tô màu đồ thị | Assignment / Labeling |
| 1.2 | N-Queens | Assignment / Labeling |
| 2.1 | Job-shop scheduling | Scheduling — machine |
| 2.2 | Employee / Shift scheduling | Scheduling — rostering |
| 3.1 | Lập lịch thi đấu round-robin | Scheduling / Sequencing |
| 3.2 | Thời khoá biểu có ràng buộc khả dụng | Scheduling — timetabling |


## Môi trường thực thi

Ô dưới đây kiểm tra cả ba engine có gọi được không. Nó chạy thật một model nhỏ
trên từng engine chứ không chỉ kiểm tra sự tồn tại của thư viện.

In [1]:
import subprocess, sys, pathlib
ROOT = pathlib.Path.cwd().parent
print(subprocess.run([sys.executable, str(ROOT / "tools" / "check_env.py")],
                     capture_output=True, text=True, cwd=ROOT).stdout)

Kiểm tra ba chiều của báo cáo:

[  OK  ] OPL          oplrun — engine CP Optimizer
         /mnt/d/Program Files/IBM/ILOG/CPLEX_Studio_Community222/opl/bin/x64_win64/oplrun.exe
[  OK  ] DOcplex.cp   docplex 2.32.264 — engine CP Optimizer
         /mnt/d/Program Files/IBM/ILOG/CPLEX_Studio_Community222/cpoptimizer/bin/x64_win64/cpoptimizer.exe
[  OK  ] OR-Tools     ortools 9.14.6206 — engine CP-SAT




### Ghi chú về bản Community Edition

CPLEX Optimization Studio dùng ở đây là bản **Community**, giới hạn CP Optimizer ở
không gian tìm kiếm $2^{1000}$. Hai điều cần biết khi đọc log:

1. Dòng `Problem size limit exceeded.` **luôn** xuất hiện trong banner giấy phép,
   kể cả khi model giải xong bình thường. Nó **không** phải báo lỗi.
2. Tín hiệu chạm trần thật là `FATAL[ENGINE_001]` kèm `### ENGINE exception`.

Giới hạn này ràng buộc cách mã hoá chứ không ràng buộc cỡ bài: với mã hoá nhị
phân, $\log_2$ không gian tìm kiếm bằng đúng **số biến bool**, nên bài vài nghìn
biến bool sẽ vượt trần; còn mã hoá bằng biến nguyên miền nhỏ thì cùng bài đó lại
lọt thoải mái. Bài 3.2 phân tích kỹ điểm này.

## Bảng nguồn — bài nào lấy mẫu chính thức, bài nào viết mới

Một dụng ý của báo cáo là cho thấy **khoảng trống của từng nền tảng**: bài nào nền
tảng đã có ví dụ chính thức, bài nào phải tự viết. Bảng dưới đây sinh thẳng từ các
file `manifest.json` trong `models/`, **không gõ tay**, nên không bao giờ lệch với
code thật đang có trong kho.

In [2]:
import sys; sys.path.insert(0, "../tools")
from nbutil import show_source_matrix
show_source_matrix()

| Bài | Tên | OPL | DOcplex.cp | OR-Tools |
|---|---|---|---|---|
| 1.1 | Tô màu đồ thị (Graph Coloring) | ✅ | ✅ | ✍️ |
| 1.2 | N-Queens | ✍️ | ✅ | ✅ |
| 2.1 | Job-shop scheduling (instance ft06) | ✅ | ✅ | ✍️ |
| 2.2 | Xếp ca nhân sự có nguyện vọng (Employee / Shift Scheduling with shift requests) | ✍️ | ✍️ | ✅ |
| 3.1 | Lập lịch thi đấu thể thao (double round-robin) | ✅ | ✅ | ✍️ |
| 3.2 | Xếp thời khoá biểu có ràng buộc khả dụng | ✅+✍️ | ✍️ | ✍️ |

✅ lấy mẫu chính thức · ✍️ viết mới · ✅+✍️ mẫu chính thức có mở rộng · — chưa có